In [13]:
!pip install laya


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
import json
import pandas as pd

# ۱. خواندن دیتای تمیز ۲۴تایی که قبلاً ساختیم
records = []
with open("contract_risk_train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)

# ۲. ذخیره به فرمت استاندارد برای laya-train
# ستون‌های اصلی: text (متن بند) و label (سطح ریسک: مناسب، کم‌ریسک، پرریسک)
df.to_csv("contract_risk_train.csv", index=False, encoding="utf-8-sig")

print(f"تعداد ردیف‌ها: {len(df)}")
print("ستون‌ها:", df.columns.tolist())
print(df.head(2))


تعداد ردیف‌ها: 24
ستون‌ها: ['text', 'label']
                                                text  label
0  کارفرما متعهد است مبلغ حق‌الزحمه مندرج در قرار...      0
1  کارفرما متعهد است حق‌الزحمه را پس از تحویل و ت...      1


In [17]:
!laya-train --data contract_risk_train.csv --text-column text --label-column label --base multilingual --dry-run


dry-run: 24 rows read, 24 valid items, skipped: {}
dry-run: optimizer budget: 22 train item(s), effective batch up to 64 (micro_batch 8 x grad_accum 8), 4 optimizer updates over 4 epoch(s)



Fetching 5 files: 100%|##########| 5/5 [00:00<00:00, 181.88it/s]


In [18]:
!laya-train --data contract_risk_train.csv --text-column text --label-column label --base multilingual --micro-batch 2 --grad-accum 2 --epochs 10 --instructions "میزان ریسک این بند قرارداد فریلنسری را تعیین کن" --out ./contract-risk-model


train items 22, calibration items 2, eval items 2 (calibration_slice), skipped {}, device cpu
optimizer budget: 22 train item(s), effective batch up to 4 (micro_batch 2 x grad_accum 2), 60 optimizer updates over 10 epoch(s)
epoch 1/10 mean loss 0.9133
epoch 2/10 mean loss 0.6675
epoch 3/10 mean loss 0.4343
epoch 4/10 mean loss 0.6189
epoch 5/10 mean loss 0.2510
epoch 6/10 mean loss 0.0435
epoch 7/10 mean loss -0.0306
epoch 8/10 mean loss 0.1258
epoch 9/10 mean loss -0.1085
epoch 10/10 mean loss -0.0612

=== Calibration Evidence (in-sample calibration slice, 2 items; not held-out) ===
Metric                 Before         After          Delta         
-----------------------------------------------------------------
Accuracy               0.0000         1.0000         +1.0000       
Loss                   1.3448         0.0076         -1.3372       
ECE                    0.4822         0.0076         -0.4746       
Brier (multi-class)    0.8548         0.0002         -0.8546       
Bri

F:\narges_joon\1405\borhan\lyda\contract-risk-project\venv\Lib\site-packages\laya\train_cli.py:221: RuntimeWarning: laya.train: choice calibration: not fitted: 2 calibration items, fewer than 10, so the temperature stays 1.0
  summary = finetune(
